In [ ]:
import numpy as np
residuals = np.load('../../graph/ELAM_data_quant.npy', allow_pickle=True)
print(residuals.shape)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import torch

layerwise_data = list(zip(*residuals))
layerwise_arrays = [np.stack(layer, axis=0) for layer in layerwise_data]


# data = layerwise_arrays[4]
data = np.concatenate([emb for emb in layerwise_arrays], axis=1)
# data = np.sum([emb for emb in layerwise_arrays], axis=0)
print(data.shape)
sim = torch.tensor(cosine_similarity(data))
print(sim.shape)

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def generate_trec(np_arr, top_k=30, output_file="output.trec"):
    n = np_arr.shape[0]
    
    # Step 1: Compute cosine similarity
    cos_sim = cosine_similarity(np_arr)  # shape (n, n)
    np.fill_diagonal(cos_sim, -np.inf)   # ignore self-similarity

    # Step 2: For each row, select top-k indices
    top_indices = np.argsort(cos_sim, axis=1)[:, -top_k:][:, ::-1]  # shape (n, top_k)

    # Step 3: Write .trec file
    with open(output_file, "w") as f:
        for i in range(n):  # for each query/sample
            for rank, j in enumerate(top_indices[i]):
                # score = cos_sim[i, j]
                f.write(f"{i} Q0 {j} {rank+1}\n")

    print(f"Written top-{top_k} TREC file to: {output_file}")

generate_trec(data, top_k=30, output_file="ELAM.trec")